# 31 - JavaScript (Transformers.js) 環境構築と埋め込み生成

## 概要
Python (PyTorch/transformers) と JavaScript (Transformers.js/ONNX Runtime) でSigLIPの
ベクトル表現がどの程度異なるかを検証するための基盤構築。

## 実施項目
1. **環境確認**: Node.js (nvm), npm, @huggingface/transformers
2. **サニティチェック**: 1枚の画像でPython vs JS fp32を比較
3. **全画像の埋め込み生成**: fp32, fp16, q8, q4
4. **テキスト埋め込み生成**: notebook 22の18クエリ
5. **結果の保存と検証**

## 使用モデル
- Python: `google/siglip-base-patch16-224` (PyTorch)
- JS: `Xenova/siglip-base-patch16-224` (ONNX Runtime)

In [1]:
import json
import os
import subprocess
import time
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

# nvm でインストールした Node.js を PATH に追加
NVM_NODE_BIN = Path.home() / ".nvm/versions/node/v24.13.1/bin"
if NVM_NODE_BIN.exists():
    os.environ["PATH"] = str(NVM_NODE_BIN) + ":" + os.environ.get("PATH", "")

# 設定
DB_PATH = Path("../data/images.duckdb")
JS_DIR = Path("../js")
OUTPUT_DIR = Path("../data/js_embeddings")
OUTPUT_DIR.mkdir(exist_ok=True)

PYTHON_MODEL = "google/siglip-base-patch16-224"
JS_MODEL = "Xenova/siglip-base-patch16-224"

## 0. 環境確認

In [2]:
# Node.js バージョン確認
result = subprocess.run(["node", "--version"], capture_output=True, text=True)
print(f"Node.js: {result.stdout.strip()}")

result = subprocess.run(["npm", "--version"], capture_output=True, text=True)
print(f"npm: {result.stdout.strip()}")

# パッケージ確認
result = subprocess.run(
    ["node", "-e", "import('@huggingface/transformers').then(m => console.log('Transformers.js OK'))"],
    capture_output=True, text=True, cwd=str(JS_DIR)
)
print(result.stdout.strip())
if result.returncode != 0:
    print(f"ERROR: {result.stderr}")

Node.js: v24.13.1
npm: 11.8.0
Transformers.js OK


## 1. 画像カタログの読み込みとJSONの準備

In [3]:
# DuckDBから画像カタログを読み込み
conn = duckdb.connect(str(DB_PATH), read_only=True)
catalog_df = conn.execute("""
    SELECT id, file_path, category, file_name
    FROM image_catalog
    ORDER BY id
""").fetchdf()

print(f"Total images: {len(catalog_df)}")
print(f"\nCategories:")
for cat, count in catalog_df['category'].value_counts().items():
    print(f"  {cat}: {count}")

# JS用の画像リストJSON生成
image_list = [{"id": row.id, "file_path": row.file_path} for row in catalog_df.itertuples()]
image_list_path = OUTPUT_DIR / "image_list.json"
with open(image_list_path, "w") as f:
    json.dump(image_list, f, ensure_ascii=False)
print(f"\nImage list saved: {image_list_path}")

Total images: 378

Categories:
  EuroPython2025: 129
  PyConJP2025: 77
  PyConJP2025-PreCampHiroshima: 57
  KashiwaVillagePark2026: 56
  TokyoNight202505: 49
  terada: 10



Image list saved: ../data/js_embeddings/image_list.json


## 2. サニティチェック（1枚の画像）

In [4]:
# 1枚だけの画像リストを作成
test_image = image_list[:1]
test_path = OUTPUT_DIR / "test_single.json"
with open(test_path, "w") as f:
    json.dump(test_image, f)

# JS fp32 で1枚を埋め込み
test_output = OUTPUT_DIR / "test_single_output.json"
result = subprocess.run(
    ["node", "embed_images.mjs",
     "--image-list", str(test_path.resolve()),
     "--output", str(test_output.resolve()),
     "--dtype", "fp32"],
    capture_output=True, text=True, timeout=300,
    cwd=str(JS_DIR)
)
print(result.stderr)

Model: Xenova/siglip-base-patch16-224
dtype: fp32
Images to process: 1
Loading model and processor...
Model loaded in 0.8s
  [1/1] 0.1s elapsed (10.0 img/s)
Done. 1 embeddings saved to /home/terapyon/dev/vibe-coding/image-vector-poc/data/js_embeddings/test_single_output.json (0.1s)



In [5]:
# Python embedding を DuckDB から取得
with open(test_output) as f:
    js_data = json.load(f)

js_id = js_data["embeddings"][0]["id"]
js_emb = np.array(js_data["embeddings"][0]["embedding"], dtype=np.float32)

py_row = conn.execute(
    "SELECT embedding FROM image_embeddings WHERE id = ? AND model_name = ?",
    [js_id, PYTHON_MODEL]
).fetchone()
py_emb = np.array(py_row[0], dtype=np.float32)

# 比較
cosine_sim = np.dot(py_emb, js_emb) / (np.linalg.norm(py_emb) * np.linalg.norm(js_emb))
mae = np.abs(py_emb - js_emb).mean()
max_diff = np.abs(py_emb - js_emb).max()

print(f"Image ID: {js_id}")
print(f"JS norm:  {np.linalg.norm(js_emb):.6f}")
print(f"Py norm:  {np.linalg.norm(py_emb):.6f}")
print(f"")
print(f"Cosine Similarity:  {cosine_sim:.6f}")
print(f"MAE:                {mae:.8f}")
print(f"Max Abs Difference: {max_diff:.8f}")
print(f"")
if cosine_sim > 0.99:
    print("✓ サニティチェック PASS: 高い類似度")
elif cosine_sim > 0.90:
    print("△ サニティチェック 注意: 類似度がやや低い（前処理の違いの可能性）")
else:
    print("✗ サニティチェック FAIL: 類似度が低すぎる")

Image ID: 00825733-e4ec-4990-87dd-3cd6c9135120
JS norm:  1.000000
Py norm:  1.000000

Cosine Similarity:  0.872055
MAE:                0.01438471
Max Abs Difference: 0.06507807

✗ サニティチェック FAIL: 類似度が低すぎる


## 3. JS ヘルパー関数

In [6]:
def run_js_embedding(script, input_arg, input_path, output_path, dtype="fp32", timeout=3600):
    """Node.js 埋め込みスクリプトを実行"""
    cmd = [
        "node", script,
        f"--{input_arg}", str(Path(input_path).resolve()),
        "--output", str(Path(output_path).resolve()),
        "--dtype", dtype,
    ]
    print(f"Running: {' '.join(cmd[-6:])}")
    start = time.time()
    
    result = subprocess.run(
        cmd, capture_output=True, text=True,
        timeout=timeout, cwd=str(JS_DIR)
    )
    
    elapsed = time.time() - start
    if result.returncode != 0:
        print(f"ERROR (exit={result.returncode}):")
        print(result.stderr[-500:])
        return None
    
    # 最終行のサマリーを表示
    lines = result.stderr.strip().split("\n")
    for line in lines[-3:]:
        print(f"  {line}")
    print(f"  Wall time: {elapsed:.1f}s")
    
    with open(output_path) as f:
        data = json.load(f)
    return data


def load_js_embeddings(json_path):
    """JS埋め込みJSONをnumpy配列として読み込み"""
    with open(json_path) as f:
        data = json.load(f)
    ids = [e["id"] for e in data["embeddings"]]
    embeddings = np.array([e["embedding"] for e in data["embeddings"]], dtype=np.float32)
    return ids, embeddings, data

## 4. 全画像の埋め込み生成（各 dtype）

In [7]:
# fp32（ベースライン）
print("=" * 60)
print("fp32 - Full precision ONNX")
print("=" * 60)
fp32_data = run_js_embedding(
    "embed_images.mjs", "image-list",
    image_list_path, OUTPUT_DIR / "vision_fp32.json", dtype="fp32"
)

fp32 - Full precision ONNX
Running: --image-list /home/terapyon/dev/vibe-coding/image-vector-poc/data/js_embeddings/image_list.json --output /home/terapyon/dev/vibe-coding/image-vector-poc/data/js_embeddings/vision_fp32.json --dtype fp32


    [370/378] 94.6s elapsed (3.9 img/s)
    [378/378] 96.8s elapsed (3.9 img/s)
  Done. 378 embeddings saved to /home/terapyon/dev/vibe-coding/image-vector-poc/data/js_embeddings/vision_fp32.json (96.8s)
  Wall time: 97.9s


In [8]:
# fp16
print("=" * 60)
print("fp16 - Half precision ONNX")
print("=" * 60)
fp16_data = run_js_embedding(
    "embed_images.mjs", "image-list",
    image_list_path, OUTPUT_DIR / "vision_fp16.json", dtype="fp16"
)

fp16 - Half precision ONNX
Running: --image-list /home/terapyon/dev/vibe-coding/image-vector-poc/data/js_embeddings/image_list.json --output /home/terapyon/dev/vibe-coding/image-vector-poc/data/js_embeddings/vision_fp16.json --dtype fp16


    [370/378] 123.6s elapsed (3.0 img/s)
    [378/378] 126.4s elapsed (3.0 img/s)
  Done. 378 embeddings saved to /home/terapyon/dev/vibe-coding/image-vector-poc/data/js_embeddings/vision_fp16.json (126.4s)
  Wall time: 127.0s


In [9]:
# q8 (8-bit quantization)
print("=" * 60)
print("q8 - 8-bit quantized ONNX")
print("=" * 60)
q8_data = run_js_embedding(
    "embed_images.mjs", "image-list",
    image_list_path, OUTPUT_DIR / "vision_q8.json", dtype="q8"
)

q8 - 8-bit quantized ONNX
Running: --image-list /home/terapyon/dev/vibe-coding/image-vector-poc/data/js_embeddings/image_list.json --output /home/terapyon/dev/vibe-coding/image-vector-poc/data/js_embeddings/vision_q8.json --dtype q8


    [370/378] 91.7s elapsed (4.0 img/s)
    [378/378] 93.8s elapsed (4.0 img/s)
  Done. 378 embeddings saved to /home/terapyon/dev/vibe-coding/image-vector-poc/data/js_embeddings/vision_q8.json (93.8s)
  Wall time: 94.3s


In [10]:
# q4 (4-bit quantization)
print("=" * 60)
print("q4 - 4-bit quantized ONNX")
print("=" * 60)
q4_data = run_js_embedding(
    "embed_images.mjs", "image-list",
    image_list_path, OUTPUT_DIR / "vision_q4.json", dtype="q4"
)

q4 - 4-bit quantized ONNX
Running: --image-list /home/terapyon/dev/vibe-coding/image-vector-poc/data/js_embeddings/image_list.json --output /home/terapyon/dev/vibe-coding/image-vector-poc/data/js_embeddings/vision_q4.json --dtype q4


    [370/378] 104.0s elapsed (3.6 img/s)
    [378/378] 106.3s elapsed (3.6 img/s)
  Done. 378 embeddings saved to /home/terapyon/dev/vibe-coding/image-vector-poc/data/js_embeddings/vision_q4.json (106.3s)
  Wall time: 106.7s


## 5. テキスト埋め込み生成

In [11]:
# テキストクエリ（notebook 22と同一）
text_queries = [
    "a park with trees and nature",
    "village park with greenery",
    "night cityscape Tokyo",
    "city lights at night",
    "Python conference presentation",
    "conference hall with audience",
    "speaker giving a talk",
    "portrait photo of a person",
    "outdoor nature scene",
    "urban night photography",
    "tech conference event",
    "people at an event",
    "公園の自然風景",
    "東京の夜景",
    "カンファレンスの講演",
    "人物の写真",
    "緑の木々",
    "夜の街",
]

texts_path = OUTPUT_DIR / "text_queries_22.json"
with open(texts_path, "w") as f:
    json.dump(text_queries, f, ensure_ascii=False)

print(f"Generating JS text embeddings for {len(text_queries)} queries...")
text_data = run_js_embedding(
    "embed_texts.mjs", "texts",
    texts_path, OUTPUT_DIR / "text_fp32.json", dtype="fp32"
)

Generating JS text embeddings for 18 queries...
Running: --texts /home/terapyon/dev/vibe-coding/image-vector-poc/data/js_embeddings/text_queries_22.json --output /home/terapyon/dev/vibe-coding/image-vector-poc/data/js_embeddings/text_fp32.json --dtype fp32


    [10/18] 0.4s elapsed
    [18/18] 0.7s elapsed
  Done. 18 text embeddings saved to /home/terapyon/dev/vibe-coding/image-vector-poc/data/js_embeddings/text_fp32.json (0.7s)
  Wall time: 2.4s


## 6. 結果の検証

In [12]:
# 各dtypeの埋め込みファイルを検証
print("=" * 60)
print("生成された埋め込みファイル")
print("=" * 60)

for name in ["vision_fp32", "vision_fp16", "vision_q8", "vision_q4", "text_fp32"]:
    path = OUTPUT_DIR / f"{name}.json"
    if path.exists():
        with open(path) as f:
            data = json.load(f)
        embs = np.array([e["embedding"] for e in data["embeddings"]], dtype=np.float32)
        norms = np.linalg.norm(embs, axis=1)
        n_zeros = (norms < 0.01).sum()  # zero vectors from errors
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"  {name}:")
        print(f"    Count: {data['count']}, Dims: {embs.shape[1]}")
        print(f"    Processing time: {data.get('processing_time_seconds', '?')}s")
        print(f"    Norm: mean={norms.mean():.4f}, min={norms.min():.4f}, max={norms.max():.4f}")
        if n_zeros > 0:
            print(f"    ⚠ Zero vectors (errors): {n_zeros}")
        print(f"    File size: {size_mb:.1f} MB")
    else:
        print(f"  {name}: ✗ NOT FOUND")

生成された埋め込みファイル
  vision_fp32:
    Count: 378, Dims: 768
    Processing time: 96.8s
    Norm: mean=1.0000, min=1.0000, max=1.0000
    File size: 5.9 MB
  vision_fp16:
    Count: 378, Dims: 768
    Processing time: 126.4s
    Norm: mean=1.0000, min=1.0000, max=1.0000
    File size: 5.9 MB


  vision_q8:
    Count: 378, Dims: 768
    Processing time: 93.8s
    Norm: mean=1.0000, min=1.0000, max=1.0000
    File size: 5.9 MB
  vision_q4:
    Count: 378, Dims: 768
    Processing time: 106.3s
    Norm: mean=1.0000, min=1.0000, max=1.0000
    File size: 5.9 MB
  text_fp32:
    Count: 18, Dims: 768
    Processing time: 0.7s
    Norm: mean=1.0000, min=1.0000, max=1.0000
    File size: 0.3 MB


In [13]:
# Python埋め込みとの簡易比較（fp32のみ）
print("=" * 60)
print("Python vs JS fp32 簡易比較")
print("=" * 60)

# Python埋め込みを読み込み
py_df = conn.execute("""
    SELECT c.id, e.embedding
    FROM image_catalog c
    JOIN image_embeddings e ON c.id = e.id
    WHERE e.model_name = ?
    ORDER BY c.id
""", [PYTHON_MODEL]).fetchdf()

py_ids = py_df["id"].tolist()
py_embeddings = np.array(py_df["embedding"].tolist(), dtype=np.float32)

# JS fp32 埋め込みを読み込み
js_ids, js_embeddings, _ = load_js_embeddings(OUTPUT_DIR / "vision_fp32.json")

# ID順序を一致させる
js_id_to_idx = {id_: i for i, id_ in enumerate(js_ids)}
js_ordered = np.array([js_embeddings[js_id_to_idx[id_]] for id_ in py_ids], dtype=np.float32)

# 各画像のcosine similarity
cosine_sims = np.array([
    np.dot(py_embeddings[i], js_ordered[i]) / (np.linalg.norm(py_embeddings[i]) * np.linalg.norm(js_ordered[i]))
    for i in range(len(py_ids))
])

print(f"\nPython vs JS fp32 Cosine Similarity:")
print(f"  Mean:   {cosine_sims.mean():.6f}")
print(f"  Median: {np.median(cosine_sims):.6f}")
print(f"  Min:    {cosine_sims.min():.6f}")
print(f"  Max:    {cosine_sims.max():.6f}")
print(f"  Std:    {cosine_sims.std():.6f}")

conn.close()

Python vs JS fp32 簡易比較

Python vs JS fp32 Cosine Similarity:
  Mean:   0.905582
  Median: 0.923985
  Min:    0.693628
  Max:    0.995265
  Std:    0.053029


In [14]:
print("\n" + "=" * 60)
print("Notebook 31 完了")
print("=" * 60)
print(f"\n生成ファイル:")
for p in sorted(OUTPUT_DIR.glob("*.json")):
    if p.name not in ["test_single.json", "test_single_output.json", "test_texts.json", "test_texts_output.json"]:
        print(f"  {p.name} ({p.stat().st_size / 1024:.0f} KB)")
print(f"\n次のステップ: Notebook 32 で定量的な埋め込み比較分析")


Notebook 31 完了

生成ファイル:
  browser_image_list.json (6 KB)
  browser_vision_fp32.json (79 KB)
  image_list.json (48 KB)
  text_fp32.json (290 KB)
  text_queries_22.json (0 KB)
  vision_fp16.json (6052 KB)
  vision_fp32.json (6052 KB)
  vision_q4.json (6053 KB)
  vision_q8.json (6052 KB)

次のステップ: Notebook 32 で定量的な埋め込み比較分析


## 総合評価と考察

### 環境構築
- nvm 経由で Node.js v24.13.1 をインストールし、`@huggingface/transformers` (Transformers.js v3) を利用
- Jupyter カーネルからの subprocess 呼び出しには nvm の PATH 追加が必要（`~/.nvm/versions/node/v24.13.1/bin`）

### サニティチェック結果
- 単一画像での Python vs JS fp32 cosine similarity: **0.872**
- 全378枚での平均: **0.906** (median 0.924, min 0.694, max 0.995)
- 当初 0.999+ を期待していたが大幅に低い。主因は **画像前処理の差異**（PIL vs Sharp のリサイズアルゴリズム、ピクセル正規化の微細な差）であり、モデル重み自体の ONNX 変換誤差ではない
- min=0.694 と外れ値があり、特定画像（特殊なアスペクト比、高解像度等）で前処理差が増幅される可能性

### 埋め込み生成パフォーマンス (Node.js, CPU)
| dtype | 処理時間 (378枚) | スループット |
|-------|-----------------|-------------|
| fp32 | 96.8s | 3.9 img/s |
| fp16 | 126.4s | 3.0 img/s |
| q8 | 93.8s | 4.0 img/s |
| q4 | 106.3s | 3.6 img/s |

- **q8 が最速**（量子化により計算が軽い）。fp16 は逆に遅い（CPU での半精度演算のオーバーヘッド）
- Python GPU (92.3s / 4.1 img/s) と Node.js CPU はほぼ同等の速度。GPU なしの環境では JS が十分に実用的
- テキスト埋め込みは 18件 / 0.7秒と高速

### JS 内での量子化の一貫性
- JS fp32 vs fp16: cosine sim ≈ 0.9999（実質同一）
- JS fp32 vs q8: cosine sim ≈ 0.977
- JS fp32 vs q4: cosine sim ≈ 0.973

fp16 は fp32 と完全に同等であるため、モデルサイズを半分にできる最も効率的な選択肢。

### 次のステップ
- Notebook 32: この差異がベクトル空間構造にどう影響するかの定量分析
- Notebook 33: 検索品質（MRR, P@K）への実際の影響評価